In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn



device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
class DFL_GMVPLossFunction(torch.autograd.Function):

  @staticmethod
  def forward(ctx,
              cov_pred:torch.Tensor,
              cov_true:torch.Tensor,
              ):
    """
      Args:
          cov_true: (batch_size, num_assets, num_assets) の真の共分散行列
          cov_pred: (batch_size, num_assets, num_assets) の予測された共分散行列
    """
    B,N,_=cov_true.shape
    ones=torch.ones((B,N,1),device=cov_true.device,dtype=cov_true.dtype)

    pred_inv_ones=torch.linalg.solve(cov_pred,ones)
    true_inv_ones=torch.linalg.solve(cov_true,ones)

    c_pred=ones.transpose(1,2)@pred_inv_ones
    c_true=ones.transpose(1,2)@true_inv_ones

    w_pred=pred_inv_ones/c_pred
    w_true=true_inv_ones/c_true

    var_pred=w_pred.transpose(1,2)@cov_true@w_pred
    var_true=w_true.transpose(1,2)@cov_true@w_true

    loss_per_batch=(var_pred-var_true).squeeze()
    loss=loss_per_batch.mean()

    ctx.save_for_backward(cov_pred,cov_true,w_pred,c_pred)
    return loss

  @staticmethod
  def backward(ctx,grad_output):
    cov_pred,cov_true,w_pred,c_pred=ctx.saved_tensors
    B,N,_=cov_pred.shape
    g=2.0*torch.bmm(cov_true,w_pred)

    pred_inv_g=torch.linalg.solve(cov_pred,g)
    w_g_dot=w_pred.transpose(1,2)@g

    term1=-torch.bmm(pred_inv_g,w_pred.transpose(1,2))
    term2=c_pred*w_g_dot*torch.bmm(w_pred,w_pred.transpose(1,2))
    M=term1+term2

    grad_cov_pred=0.5*(M+M.transpose(1,2))
    grad_cov_pred=grad_cov_pred*(grad_output/B)
    return grad_cov_pred,None


class DFL_GMVPLoss(nn.Module):
  def __init__(self):
    super(DFL_GMVPLoss,self).__init__()
  def forward(self,cov_pred,cov_true):
    return DFL_GMVPLossFunction.apply(cov_pred,cov_true)


class DFL_DLinear(nn.Module):
  def __init__(self,in_dim,num_assets,hidden_dim=128):
    super(DFL_DLinear,self).__init__()
    self.in_dim=in_dim
    self.num_assets=num_assets
    self.K=(num_assets+1)*num_assets//2

    self.flattened_dim=self.num_assets*self.in_dim
    self.fc_seasonal=nn.Sequential(
        nn.Linear(self.flattened_dim,hidden_dim),
        nn.Linear(hidden_dim,self.K)
    )
    self.fc_trend=nn.Sequential(
        nn.Linear(self.flattened_dim,hidden_dim),
        nn.Linear(hidden_dim,self.K)
    )

    kernel_size=max(5,min(in_dim//3,50))
    if kernel_size%2==0: kernel_size+=1
    self.trend_pool=nn.AvgPool1d(kernel_size=kernel_size,stride=1,padding=kernel_size//2)


  def forward(self,x):
    # x.shape=(batch_size,num_assets,in_dim)
    batch_size=x.size(0)

    x_trend=self.trend_pool(x)
    x_seasonal=x-x_trend

    x_trend_flat=x_trend.view(batch_size,-1)
    x_seasonal_flat=x_seasonal.view(batch_size,-1)

    h_seasonal=self.fc_seasonal(x_seasonal_flat)
    h_trend=self.fc_trend(x_trend_flat)
    h=h_trend+h_seasonal
    L=self._build_lower_triangular(h)
    return torch.bmm(L,L.transpose(1,2))


  def _build_lower_triangular(self,h):
    batch_size=h.shape[0]
    L=torch.zeros(batch_size,self.num_assets,self.num_assets,device=h.device)
    row_index,col_index=torch.tril_indices(self.num_assets,self.num_assets)
    L[:,row_index,col_index]=h

    diag_idx=torch.arange(self.num_assets)
    epsilon=1e-6
    L[:,diag_idx,diag_idx]=nn.functional.softplus(L[:,diag_idx,diag_idx])+epsilon
    return L